# Camada Silver - Tratamento dos Dados

## Objetivo

A camada Silver é responsável por realizar o tratamento dos dados brutos provenientes da camada Bronze, garantindo maior qualidade, consistência e padronização antes da disponibilização para consumo analítico na camada Gold.

## Transformações Aplicadas

#### Tabela de Transações PIX

- Leitura dos dados da tabela `bronze.pix_transactions`;
- Seleção apenas das colunas relevantes para análise;
- Substituição de valores nulos por `0`;
- Remoção de registros duplicados;
- Extração do ano e mês a partir da coluna `AnoMes`;
- Remoção da coluna original `AnoMes`.

#### Tabela de Estatísticas de Pagamentos

- Leitura dos dados da tabela `bronze.payment_statistics`;
- Substituição de valores nulos por `0`;
- Remoção de registros duplicados;
- Extração do ano e mês a partir da coluna `AnoMes`;
- Remoção da coluna original `AnoMes`.

## Tabelas Geradas

Após os tratamentos, os dados são armazenados em formato Delta Lake nas seguintes tabelas:

- `silver.pix_transactions`
- `silver.payment_statistics`

## Benefícios da Camada Silver

- Melhora a qualidade e confiabilidade dos dados;
- Elimina registros duplicados;
- Padroniza a estrutura temporal para análises futuras;
- Facilita a criação de métricas e agregações na camada Gold;
- Disponibiliza dados limpos e consistentes para consumo analítico.

In [0]:
from pyspark.sql.functions import substring

#Leitura da tabela de estatísticas de pagamentos
payment_df = (
    spark.table("bronze.payment_statistics")
    .na.fill(0)
    .dropDuplicates()
)

#Leitura da tabela de transações do PIX
pix_df = (
    spark.table("bronze.pix_transactions")
    .select(
        "AnoMes",
        "Estado",
        "Municipio",
        "Regiao",
        "VL_PagadorPF",
        "VL_PagadorPJ",
        "VL_RecebedorPF",
        "VL_RecebedorPJ"
    )
    .na.fill(0)
    .dropDuplicates()
)

#Transformações da tabela de transações do PIX
    # - Extração do ano com base nos 4 primeiros caracteres de AnoMes
    # - Extração do mês com base nos 2 últimos caracteres de AnoMes
silver_pix = (
    pix_df
    .withColumn("ano", substring("AnoMes", 1, 4))
    .withColumn("mes", substring("AnoMes", 5, 2))
).drop("AnoMes")


#Transformação da tabela de pagamentos
    # - Extração do ano com base nos 4 primeiros caracteres de AnoMes
    # - Extração do mês com base nos 2 últimos caracteres de AnoMes
silver_payment = (
    payment_df
    .withColumn("ano", substring("AnoMes", 1, 4))
    .withColumn("mes", substring("AnoMes", 5, 2))
).drop("AnoMes")

#Gravação na tabela silver com os devidos tratamentos realizados
silver_pix.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.pix_transactions")

silver_payment.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.payment_statistics")